# Discover Model Settings

Probe the configured LM provider for the value space of `reasoning_effort`
and pick the lowest-reasoning value the model actually honors. Persists
the result to `sessions/<model-slug>/model_settings.ipynb`.

Uses raw HTTP only — must work *before* the agent has good LM settings.

In [ ]:
# parameters
task_id = None
parent_task_id = None
input_payload = {}
output_dir = "./outputs"
run_dir = "."
budget = {}
target_model = None
base_url = None
api_key = None
candidates = [None, "off", "low", "medium", "high"]
canary_prompt = "Reply with exactly the single word OK and nothing else."
per_request_timeout = 30.0
sessions_root = "sessions"

In [ ]:
from pathlib import Path
from notebook_agent.litellm_client import LiteLLMClient
from notebook_agent.model_settings import (
    pick_loaded_model,
    settings_notebook_path,
    write_settings,
)
from notebook_agent.notebook_init import get_notebook_config

Path(output_dir).mkdir(parents=True, exist_ok=True)

# Resolve the client: explicit params win, then the active notebook config,
# then env defaults.
_cfg_client = None
try:
    _cfg_client = get_notebook_config().client
except Exception:
    _cfg_client = None

_client = LiteLLMClient(
    base_url=base_url or (_cfg_client.base_url if _cfg_client else None),
    api_key=api_key or (_cfg_client.api_key if _cfg_client else None),
    model=target_model or (_cfg_client.model if _cfg_client else None),
    provider=(_cfg_client.provider if _cfg_client else None),
    # Don't send reasoning_effort during discovery itself.
    reasoning_effort=None,
)

# Detect what's actually loaded; prefer the configured model.
_loaded = pick_loaded_model(_client, prefer=_client.model)
_probe_model = _loaded.id if _loaded else (target_model or _client.model)
if _probe_model is None:
    raise RuntimeError("No model loaded on the provider and no target_model given")
print(f"Probing model: {_probe_model!r} on {_client.base_url}")

In [ ]:
import json, time
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def _strip(model):
    return model.split("/", 1)[1] if "/" in model and model.split("/", 1)[0] in {
        "lm_studio", "openai", "anthropic", "ollama"
    } else model


def _probe(effort):
    """Send one canary request with the given reasoning_effort.

    Returns a result dict. Never raises — failures are recorded.
    """
    payload = {
        "model": _strip(_probe_model),
        "messages": [{"role": "user", "content": canary_prompt}],
        "temperature": 0.0,
        "max_tokens": 64,
    }
    if effort is not None:
        payload["reasoning_effort"] = effort
    body = json.dumps(payload).encode("utf-8")
    headers = {"Content-Type": "application/json"}
    if _client.api_key:
        headers["Authorization"] = f"Bearer {_client.api_key}"
    url = _client.base_url.rstrip("/") + "/chat/completions"
    req = Request(url, data=body, headers=headers, method="POST")
    t0 = time.monotonic()
    try:
        with urlopen(req, timeout=per_request_timeout) as resp:  # noqa: S310
            data = json.loads(resp.read().decode("utf-8"))
        elapsed = round(time.monotonic() - t0, 3)
        usage = data.get("usage") or {}
        details = usage.get("completion_tokens_details") or {}
        content = (data.get("choices") or [{}])[0].get("message", {}).get("content", "")
        return {
            "reasoning_effort": effort,
            "ok": True,
            "elapsed_s": elapsed,
            "completion_tokens": usage.get("completion_tokens"),
            "reasoning_tokens": details.get("reasoning_tokens"),
            "content_preview": (content or "")[:64],
            "http_status": 200,
        }
    except HTTPError as e:
        elapsed = round(time.monotonic() - t0, 3)
        try:
            err_body = e.read().decode("utf-8", errors="replace")[:500]
        except Exception:
            err_body = ""
        return {
            "reasoning_effort": effort,
            "ok": False,
            "elapsed_s": elapsed,
            "http_status": e.code,
            "error": err_body or str(e),
        }
    except (URLError, TimeoutError, OSError) as e:
        elapsed = round(time.monotonic() - t0, 3)
        return {
            "reasoning_effort": effort,
            "ok": False,
            "elapsed_s": elapsed,
            "http_status": None,
            "error": f"{type(e).__name__}: {e}",
        }


probes = []
for c in candidates:
    print(f"  probe reasoning_effort={c!r} ...", end=" ", flush=True)
    r = _probe(c)
    probes.append(r)
    if r["ok"]:
        print(f"ok ({r['elapsed_s']}s, reasoning_tokens={r.get('reasoning_tokens')})")
    else:
        print(f"failed ({r.get('http_status')}: {r.get('error', '')[:80]})")

In [ ]:
# Pick a winner:
# 1. Among successful probes, pick the one with the smallest reasoning_tokens
#    (treating None as +inf so a baseline-no-field is only chosen if
#    everything else failed). Ties broken by elapsed time.
# 2. If all probes succeed with similar reasoning_tokens, the model is
#    ignoring reasoning_effort — record supports_reasoning_effort=False
#    and use None (don't send the field).

_successes = [p for p in probes if p["ok"]]
if not _successes:
    raise RuntimeError(f"All probes failed: {probes!r}")

def _rt(p):
    v = p.get("reasoning_tokens")
    return v if isinstance(v, int) else 10**9

_min_rt = min(_rt(p) for p in _successes)
_max_rt = max(_rt(p) for p in _successes)
# "Honored" = sending a value visibly reduces reasoning_tokens vs sending nothing.
_baseline = next((p for p in _successes if p["reasoning_effort"] is None), None)
_baseline_rt = _rt(_baseline) if _baseline else None

supports = False
if _baseline_rt is not None and _min_rt < _baseline_rt:
    supports = True
elif _baseline is None and _max_rt - _min_rt > 10:
    supports = True

if supports:
    # Pick the candidate (not None) with the smallest reasoning_tokens.
    _named = [p for p in _successes if p["reasoning_effort"] is not None]
    _named.sort(key=lambda p: (_rt(p), p["elapsed_s"]))
    recommended = _named[0]["reasoning_effort"]
else:
    recommended = None

result = {
    "model": _probe_model,
    "base_url": _client.base_url,
    "reasoning_effort": recommended,
    "supports_reasoning_effort": supports,
    "probes": probes,
}
print("\nRecommended:", recommended, "  (supports_reasoning_effort=", supports, ")")

In [ ]:
# Persist as a Papermill notebook at sessions/<model-slug>/model_settings.ipynb.
_settings_path = settings_notebook_path(_probe_model, sessions_root=sessions_root)
_settings = {
    "model": _probe_model,
    "base_url": _client.base_url,
    "reasoning_effort": recommended,
    "supports_reasoning_effort": supports,
}
_notes_lines = ["## Probe results", ""]
for p in probes:
    _notes_lines.append(
        f"- `reasoning_effort={p['reasoning_effort']!r}` \u2192 "
        f"ok={p['ok']}, elapsed={p.get('elapsed_s')}s, "
        f"reasoning_tokens={p.get('reasoning_tokens')}"
    )
write_settings(_settings_path, _settings, notes="\n".join(_notes_lines))
result["settings_path"] = str(_settings_path)
print("Wrote", _settings_path)

In [ ]:
import json
from pathlib import Path
_out = Path(output_dir)
_out.mkdir(parents=True, exist_ok=True)
(_out / "result.json").write_text(json.dumps(result, indent=2))

_mp = Path(run_dir) / "manifest.json"
if _mp.exists():
    _m = json.loads(_mp.read_text())
    _m.setdefault("outputs", {})["result_json"] = str(_out / "result.json")
    _m.setdefault("notebook", {})["skill_id"] = "core.discover_model_settings"
    _mp.write_text(json.dumps(_m, indent=2))

assert isinstance(result, dict)
assert (_out / "result.json").exists()